# Tutorial 13 — 3-D filament: bending and rotational relaxation

Tutorial 12 reproduced two-dimensional flexible-fiber dynamics
(Delmotte et al. 2015). Here we exercise the **full 3-D mode** of
`FlexibleFiber`, which assigns a Rodrigues 3-vector to each bead, and
study a different elastohydrodynamic regime: a fiber initially bent in
a non-planar shape that **relaxes elastically** in a quiescent fluid
while simultaneously **rotating** under gravity-induced torques.

The setup mirrors the rotational dynamics of driven filaments
(Wiggins, Riveline, Ott & Goldstein, *Biophys. J.* **74**, 1043 (1998);
Coq, Du Roure, Marthelot, Bartolo & Fermigier, *Phys. Fluids* **20**,
051703 (2008)) but uses a free, gravity-loaded fiber rather than an
externally rotated tip — the latter requires kinematic boundary
constraints not currently exposed by `FlexibleFiber`.

Two effects appear together:

* **bending relaxation** — the discrete biharmonic spring drives the
  bent shape back toward the straight equilibrium;
* **rotation under gravity** — out-of-plane gravitational torques on
  the bent shape rotate the body until its plane contains the gravity
  vector.


In [1]:
import jax.numpy as jnp
import numpy as np
import plotly.graph_objects as go

from softmobility import FlexibleFiber, FlowBodyRollout, no_flow, gravity_field

np.set_printoptions(precision=3, suppress=True)


## Build a 3-D fiber

Six beads, full 3-D bending (`planar=False`, three Rodrigues components
per bead, so `Ndof = 18`). Bond half-length is `a + εg = 1.05`. Bending
modulus and mass are chosen so the elastic time and the gravitational
time are well separated and the rollout stays in the linear regime.


In [2]:
N = 6
a = 1.0

fiber = FlexibleFiber(
    n_beads=N, radius=a, gap_ratio=0.05,
    bending_rigidity=20.0, mass=0.5, planar=False,
)
rollout = FlowBodyRollout(
    soft_body=fiber,
    flow=no_flow(),
    input_map={"gravity": gravity_field(g=1.0)},
)

L = (N - 1) * 2 * a * 1.05
print(f"L = {L:.2f},  N_dof = {fiber.Ndof}")


L = 10.50,  N_dof = 18


## Initial 3-D bent shape

We perturb every bead's Rodrigues vector with a small 3-D rotation: a
y-component (planar bend) plus an x-component (twist out of the
xz-plane). The combination produces a helical-looking initial
configuration. With non-zero gap and modest amplitudes (≤ 0.4 rad), the
fiber stays free of overlaps.


In [3]:
init_dofs = jnp.zeros(fiber.Ndof)
amp = 0.30
for i in range(N):
    angle = (i - (N - 1) / 2.0) / (N - 1)   # ranges over [-0.5, 0.5]
    init_dofs = init_dofs.at[3 * i + 0].set(amp * angle)         # twist (around ê_x)
    init_dofs = init_dofs.at[3 * i + 1].set(amp * np.sin(np.pi * angle))   # bend (around ê_y)
print("initial DOFs:\n", np.asarray(init_dofs).reshape(N, 3))


initial DOFs:
 [[-0.15  -0.3    0.   ]
 [-0.09  -0.243  0.   ]
 [-0.03  -0.093  0.   ]
 [ 0.03   0.093  0.   ]
 [ 0.09   0.243  0.   ]
 [ 0.15   0.3    0.   ]]


## Run the rollout

Gravity along `-ẑ`. The fiber translates (settles) while the bending
DOFs relax and the body orientation reorients itself in space.


In [4]:
dt = 0.02
n_steps = 800

positions, orientations, dofs = rollout.rollout(
    dt=dt, n_steps=n_steps, init_dofs=init_dofs,
)
positions.block_until_ready()
print("final body z         :", float(positions[-1, 2]))
print("final body orientation:", np.asarray(orientations[-1]))
print("max |DOF| over trajectory:", float(jnp.max(jnp.abs(dofs))))
print("final |DOF|              :", float(jnp.max(jnp.abs(dofs[-1]))))


final body z         : 0.6392126083374023
final body orientation: [ 0.423 -0.424  0.516]
max |DOF| over trajectory: 1.1185882091522217
final |DOF|              : 0.81414395570755


## Convert body-frame bead positions to lab frame

Same helper used in tutorial 12.


In [5]:
def rodrigues_matrix(rvec):
    rvec = np.asarray(rvec, dtype=float)
    theta = np.linalg.norm(rvec)
    if theta < 1e-9:
        return np.eye(3)
    k = rvec / theta
    K = np.array([[0, -k[2], k[1]], [k[2], 0, -k[0]], [-k[1], k[0], 0]])
    return np.eye(3) + np.sin(theta) * K + (1 - np.cos(theta)) * (K @ K)


def lab_bead_positions(fiber, positions, orientations, dofs, design=None):
    if design is None:
        design = np.asarray(fiber.design_defaults)
    n_steps = int(positions.shape[0])
    n_beads = fiber.Nspheres
    t_dummy = np.array([0.0])
    out = np.zeros((n_steps, n_beads, 3))
    body_frame_positions = np.zeros((n_beads, 3))
    for step in range(n_steps):
        dof_step = np.asarray(dofs[step])
        for i in range(n_beads):
            body_frame_positions[i] = np.asarray(fiber.spheres[i].position(dof_step, design, t_dummy))
        R = rodrigues_matrix(orientations[step])
        out[step] = np.asarray(positions[step]) + body_frame_positions @ R.T
    return out


lab_pos = lab_bead_positions(fiber, positions, orientations, dofs)
print("lab_pos shape:", lab_pos.shape)


lab_pos shape: (800, 6, 3)


## Plot — 3-D shape over time

Snapshots in the frame moving with the centre of mass, scaled by the
fiber length. The black curve is the initial bent shape; lighter
shades show the progressive relaxation toward a straight, gravity-
aligned configuration.


In [6]:
step_indices = np.linspace(0, n_steps - 1, 6, dtype=int)
times = step_indices * dt
colors = ["#000000", "#333333", "#555555", "#777777", "#aaaaaa", "#cccccc"]

fig = go.Figure()
for c, step, t in zip(colors, step_indices, times):
    snap = lab_pos[step]
    centred = (snap - snap.mean(axis=0)) / L
    fig.add_trace(go.Scatter3d(
        x=centred[:, 0], y=centred[:, 1], z=centred[:, 2],
        mode="lines+markers",
        name=f"t = {t:.1f}",
        line=dict(color=c, width=4),
        marker=dict(size=4, color=c),
    ))

fig.update_layout(
    title=f"3-D fiber bending relaxation under gravity (N = {N})",
    scene=dict(xaxis_title="x/L", yaxis_title="y/L", zaxis_title="z/L", aspectmode="cube"),
    width=850, height=600,
)
fig.show()


## Notes

* `FlexibleFiber(planar=False)` exposes three Rodrigues components per
  bead so general 3-D bending can be initialised. The discrete
  biharmonic torque damps inter-bead bending; uniform Rodrigues
  components are *not* damped (rotating all bead frames together is
  equivalent to rigidly rotating the body), so the body-frame orientation
  may drift while the relative bending shrinks.
* For a free fiber in an **ambient shear flow**, this redundancy plus
  the singularity of the Rodrigues representation at `‖θ‖ = π` makes
  the 3-D Jeffery problem numerically delicate. Tutorial 14 uses a
  rigid two-sphere body for the clean Jeffery comparison; tutorial 12
  uses the planar mode for shear-driven dynamics.
* Reproducing the helical-actuation experiment of Coq et al. 2008
  exactly requires kinematic constraints on bead 0's position and
  orientation, currently out of scope of `FlexibleFiber`.
